In [ ]:
import vitaldb
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from scipy.signal import find_peaks, resample_poly

In [ ]:
valid_df = pd.read_csv("waveform_validation_results.csv")

In [ ]:
def extract_bp_from_abp(abp_window, fs=500):

    # -----------------------------------------
    # Basic ART quality check
    # -----------------------------------------

    if not np.isfinite(abp_window).all():
        return None, None

    # Very flat signal
    if np.ptp(abp_window) < 20:
        return None, None

    # -----------------------------------------
    # Find systolic peaks
    # -----------------------------------------

    peaks, _ = find_peaks(
        abp_window,
        distance=int(0.4 * fs),
        prominence=10
    )

    # Need at least 2 beats
    if len(peaks) < 2:
        return None, None

    # -----------------------------------------
    # SBP
    # -----------------------------------------

    sbp_values = abp_window[peaks]

    # -----------------------------------------
    # DBP
    # -----------------------------------------

    dbp_values = []

    for i in range(len(peaks) - 1):

        beat_segment = abp_window[
            peaks[i]:peaks[i + 1]
        ]

        if len(beat_segment) > 0:

            dbp_values.append(
                np.min(beat_segment)
            )

    if len(dbp_values) == 0:
        return None, None

    # -----------------------------------------
    # Median across beats
    # -----------------------------------------

    sbp = np.median(sbp_values)
    dbp = np.median(dbp_values)

    # -----------------------------------------
    # Physiological range
    # -----------------------------------------

    if not np.isfinite(sbp):
        return None, None

    if not np.isfinite(dbp):
        return None, None

    if sbp < 50 or sbp > 250:
        return None, None

    if dbp < 20 or dbp > 150:
        return None, None

    return sbp, dbp

In [ ]:
def downsample_signal(signal):

    return resample_poly(
        signal,
        up=1,
        down=4
    )

In [ ]:
def process_vitaldb_recording(
    case_id,
    WINDOW_SIZE=4000,
    STEP_SIZE=2000,
    fs=500,
    target_fs=125
):

    try:

        # =========================================
        # Load case
        # =========================================

        vf = vitaldb.VitalFile(case_id)

        tracks = vf.trks

        # =========================================
        # Check required tracks
        # =========================================

        if "SNUADC/PLETH" not in tracks:
            return None, None

        if "SNUADC/ART" not in tracks:
            return None, None

        ppg_track = tracks["SNUADC/PLETH"]
        art_track = tracks["SNUADC/ART"]

        # =========================================
        # Check sampling rates
        # =========================================

        ppg_fs = float(ppg_track.srate)
        art_fs = float(art_track.srate)

        if ppg_fs != fs:
            return None, None

        if art_fs != fs:
            return None, None

        # =========================================
        # Check records
        # =========================================

        if len(ppg_track.recs) == 0:
            return None, None

        if len(art_track.recs) == 0:
            return None, None

        # =========================================
        # Get first record
        # =========================================

        ppg_rec = ppg_track.recs[0]
        art_rec = art_track.recs[0]

        ppg = np.asarray(
            ppg_rec["val"],
            dtype=np.float32
        )

        art_raw = np.asarray(
            art_rec["val"],
            dtype=np.float32
        )

        art = (
            art_raw * float(art_track.gain)
            + float(art_track.offset)
        )

        # =========================================
        # Align using timestamps
        # =========================================

        ppg_start = float(ppg_rec["dt"])
        art_start = float(art_rec["dt"])

        start_time = max(
            ppg_start,
            art_start
        )

        ppg_start_idx = int(
            round(
                (start_time - ppg_start) * fs
            )
        )

        art_start_idx = int(
            round(
                (start_time - art_start) * fs
            )
        )

        ppg = ppg[ppg_start_idx:]
        art = art[art_start_idx:]

        # =========================================
        # Make same length
        # =========================================

        n = min(
            len(ppg),
            len(art)
        )

        ppg = ppg[:n]
        art = art[:n]

        # =========================================
        # Recording too short?
        # =========================================

        if n < WINDOW_SIZE:
            return None, None

        # =========================================
        # Normalize PPG
        # =========================================

        valid_ppg = ppg[
            np.isfinite(ppg)
        ]

        if len(valid_ppg) == 0:
            return None, None

        ppg_mean = np.mean(valid_ppg)
        ppg_std = np.std(valid_ppg)

        if ppg_std == 0 or not np.isfinite(ppg_std):
            return None, None

        ppg = (
            ppg - ppg_mean
        ) / ppg_std

        # =========================================
        # Windowing
        # =========================================

        X = []
        y = []

        for start in range(
            0,
            n - WINDOW_SIZE + 1,
            STEP_SIZE
        ):

            # -------------------------------------
            # Extract windows
            # -------------------------------------

            ppg_window = ppg[
                start:start + WINDOW_SIZE
            ]

            art_window = art[
                start:start + WINDOW_SIZE
            ]

            # -------------------------------------
            # NaN / invalid value check
            # -------------------------------------

            if not np.isfinite(ppg_window).all():
                continue

            if not np.isfinite(art_window).all():
                continue

            # -------------------------------------
            # Extract SBP / DBP
            # -------------------------------------

            sbp, dbp = extract_bp_from_abp(
                art_window,
                fs=fs
            )

            # No valid peaks / BP
            if sbp is None or dbp is None:
                continue

            # -------------------------------------
            # SBP / DBP range check
            # -------------------------------------

            if sbp < 50 or sbp > 250:
                continue

            if dbp < 20 or dbp > 150:
                continue

            # -------------------------------------
            # Downsample PPG
            # -------------------------------------

            ppg_window_125 = downsample_signal(
                ppg_window,
            )

            # -------------------------------------
            # Check expected size
            # -------------------------------------

            if len(ppg_window_125) != 1000:
                continue

            # -------------------------------------
            # Store
            # -------------------------------------

            X.append(ppg_window_125)

            y.append([
                sbp,
                dbp
            ])

        # =========================================
        # No valid windows
        # =========================================

        if len(X) == 0:
            return None, None

        # =========================================
        # Return
        # =========================================

        return (
            np.asarray(X, dtype=np.float32),
            np.asarray(y, dtype=np.float32)
        )

    except Exception as e:

        print(
            f"Error processing case {case_id}: "
            f"{type(e).__name__}: {e}"
        )

        return None, None

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np
import os


def process_single_case(args):
    """
    Wrapper so ProcessPoolExecutor can call
    process_vitaldb_recording().
    """

    case_id, WINDOW_SIZE, STEP_SIZE, fs, target_fs = args

    X, y = process_vitaldb_recording(
        case_id=case_id,
        WINDOW_SIZE=WINDOW_SIZE,
        STEP_SIZE=STEP_SIZE,
        fs=fs,
        target_fs=target_fs
    )

    return case_id, X, y


def process_vitaldb_dataset(
    valid_df,
    WINDOW_SIZE=4000,
    STEP_SIZE=2000,
    fs=500,
    target_fs=125,
    num_workers=10
):

    X_all = []
    y_all = []

    successful_cases = []
    failed_cases = []

    total_windows = 0

    case_ids = valid_df["case_id"].tolist()

    # --------------------------------------------------
    # Prepare arguments
    # --------------------------------------------------

    args_list = [
        (
            case_id,
            WINDOW_SIZE,
            STEP_SIZE,
            fs,
            target_fs
        )
        for case_id in case_ids
    ]

    # --------------------------------------------------
    # Parallel processing
    # --------------------------------------------------

    print(
        f"Processing {len(case_ids)} cases "
        f"using {num_workers} workers..."
    )

    with ProcessPoolExecutor(
        max_workers=num_workers
    ) as executor:

        futures = [
            executor.submit(
                process_single_case,
                args
            )
            for args in args_list
        ]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Processing VitalDB",
            unit="case"
        ):

            try:

                case_id, X, y = future.result()

                # --------------------------------------
                # Failed case
                # --------------------------------------

                if X is None:

                    failed_cases.append(
                        case_id
                    )

                    continue

                # --------------------------------------
                # Successful case
                # --------------------------------------

                X_all.append(X)
                y_all.append(y)

                successful_cases.append(
                    case_id
                )

                total_windows += len(X)

            except Exception as e:

                print(
                    f"\nError processing case: {e}"
                )

    # --------------------------------------------------
    # Check results
    # --------------------------------------------------

    if len(X_all) == 0:

        raise ValueError(
            "No valid windows were generated."
        )

    # --------------------------------------------------
    # Combine
    # --------------------------------------------------

    X_all = np.concatenate(
        X_all,
        axis=0
    )

    y_all = np.concatenate(
        y_all,
        axis=0
    )

    # --------------------------------------------------
    # Results
    # --------------------------------------------------

    print("\n================================")
    print("VitalDB preprocessing complete")
    print("================================")

    print(
        "Successful cases:",
        len(successful_cases)
    )

    print(
        "Failed cases:",
        len(failed_cases)
    )

    print(
        "Total windows:",
        total_windows
    )

    print(
        "X shape:",
        X_all.shape
    )

    print(
        "y shape:",
        y_all.shape
    )

    return (
        X_all,
        y_all,
        successful_cases,
        failed_cases
    )

In [ ]:
case_id = 1

X, y = process_vitaldb_recording(
    case_id=case_id,
    WINDOW_SIZE=4000,
    STEP_SIZE=2000,
    fs=500,
    target_fs=125
)

if X is None:

    print(f"Case {case_id}: NO VALID WINDOWS")

else:

    print(f"Case {case_id}: SUCCESS")

    print("\n==============================")
    print("DATASET SHAPES")
    print("==============================")

    print("X shape:", X.shape)
    print("y shape:", y.shape)

    print("\n==============================")
    print("FIRST 10 LABELS")
    print("==============================")

    print(y[:10])

    print("\n==============================")
    print("SBP STATISTICS")
    print("==============================")

    print("Min :", np.min(y[:, 0]))
    print("Max :", np.max(y[:, 0]))
    print("Mean:", np.mean(y[:, 0]))
    print("Std :", np.std(y[:, 0]))

    print("\n==============================")
    print("DBP STATISTICS")
    print("==============================")

    print("Min :", np.min(y[:, 1]))
    print("Max :", np.max(y[:, 1]))
    print("Mean:", np.mean(y[:, 1]))
    print("Std :", np.std(y[:, 1]))

Case 1: SUCCESS

DATASET SHAPES
X shape: (2481, 1000)
y shape: (2481, 2)

FIRST 10 LABELS
[[171.89493   76.1116  ]
 [169.92001   77.09909 ]
 [169.4263    76.1116  ]
 [170.9075    75.124176]
 [170.9075    75.124176]
 [169.92001   75.124176]
 [169.92001   75.124176]
 [169.4263    75.124176]
 [167.9451    75.124176]
 [166.95767   75.124176]]

SBP STATISTICS
Min : 80.06143
Max : 249.90408
Mean: 128.26375
Std : 28.79186

DBP STATISTICS
Min : 27.726212
Max : 138.32138
Mean: 58.24465
Std : 15.737224


In [ ]:
X_vitaldb, y_vitaldb, successful_cases, failed_cases = process_vitaldb_dataset(valid_df) 

Processing 3238 cases using 4 workers...


Processing VitalDB:   0%|          | 12/3238 [00:29<2:11:42,  2.45s/case]


In [ ]:
X_vitaldb, y_vitaldb, successful_cases, failed_cases = np.savez_compressed(
    "vitaldb_zero_shot_test.npz",

    X=X_vitaldb,

    y=y_vitaldb,

    case_ids=np.array(
        successful_cases
    )
)

In [ ]:
def process_vitaldb_dataset(
    valid_df,
    WINDOW_SIZE=4000,
    STEP_SIZE=2000,
    fs=500,
    target_fs=125
):

    X_all = []
    y_all = []

    successful_cases = []
    failed_cases = []

    total_windows = 0

    for case_id in tqdm(
        valid_df["case_id"].tolist(),
        desc="Processing VitalDB",
        unit="case"
    ):

        X, y = process_vitaldb_recording(
            case_id=case_id,
            WINDOW_SIZE=WINDOW_SIZE,
            STEP_SIZE=STEP_SIZE,
            fs=fs,
            target_fs=target_fs
        )

        if X is None:
            failed_cases.append(case_id)
            continue

        X_all.append(X)
        y_all.append(y)

        successful_cases.append(case_id)

        total_windows += len(X)

    # -----------------------------------------
    # Combine
    # -----------------------------------------

    if len(X_all) == 0:

        raise ValueError(
            "No valid windows were generated."
        )

    X_all = np.concatenate(
        X_all,
        axis=0
    )

    y_all = np.concatenate(
        y_all,
        axis=0
    )

    print("\n================================")
    print("VitalDB preprocessing complete")
    print("================================")

    print(
        "Successful cases:",
        len(successful_cases)
    )

    print(
        "Failed cases:",
        len(failed_cases)
    )

    print(
        "Total windows:",
        len(X_all)
    )

    print(
        "X shape:",
        X_all.shape
    )

    print(
        "y shape:",
        y_all.shape
    )

    return (
        X_all,
        y_all,
        successful_cases,
        failed_cases
    )